# DART 공시 보고서 수집
삼양그룹 계열사의 사업보고서 / 반기보고서 / 분기보고서 / 감사보고서 수집 (최대 6년)

- 수집 대상: 사업보고서(A001), 반기보고서(A002), 분기보고서(A003), 감사보고서(F001), 연결감사보고서(F002)
- API 문서: https://opendart.fss.or.kr/guide/main.do

In [ ]:
import os
import io
import zipfile
import xml.etree.ElementTree as ET
import requests
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv('DART_API_KEY')
assert API_KEY, 'DART_API_KEY가 .env에 없습니다.'
print('API Key 로드 완료')

## 1. DART corp_code 매핑 로드
DART는 종목코드 대신 자체 `corp_code`를 사용합니다.  
`corpCode.xml` zip을 받아서 회사명으로 corp_code를 찾습니다.

In [ ]:
def fetch_corp_code_df(api_key: str) -> pd.DataFrame:
    """DART 전체 기업 고유번호 목록을 DataFrame으로 반환"""
    url = f'https://opendart.fss.or.kr/api/corpCode.xml?crtfc_key={api_key}'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        with z.open('CORPCODE.xml') as f:
            tree = ET.parse(f)

    rows = []
    for item in tree.getroot().findall('list'):
        rows.append({
            'corp_code': item.findtext('corp_code'),
            'corp_name': item.findtext('corp_name'),
            'stock_code': item.findtext('stock_code'),
            'modify_date': item.findtext('modify_date'),
        })
    return pd.DataFrame(rows)

corp_df = fetch_corp_code_df(API_KEY)
print(f'전체 기업 수: {len(corp_df):,}')
corp_df.head(3)

## 2. 수집 대상 회사 정의

In [ ]:
# 수집 대상 회사 (회사명: 종목코드 or None)
TARGET_COMPANIES = {
    '삼양사':           '145990',
    '삼양패키징':       '272550',
    '삼양이노켐':       None,
    '삼양엔씨켐':       '482630',
    '삼양웰푸드':       None,
    '삼양바이오팜':     None,
    '삼양애니팜':       None,
    '삼양데이타시스템': None,
}

# 회사명으로 corp_code 매핑
def get_corp_code(corp_name: str, stock_code: str | None, df: pd.DataFrame) -> str | None:
    # 종목코드가 있으면 종목코드로 먼저 조회 (더 정확)
    if stock_code:
        matched = df[df['stock_code'] == stock_code]
        if not matched.empty:
            return matched.iloc[0]['corp_code']
    # 회사명 정확 일치
    matched = df[df['corp_name'] == corp_name]
    if not matched.empty:
        return matched.iloc[0]['corp_code']
    # 회사명 부분 일치 (fallback)
    matched = df[df['corp_name'].str.contains(corp_name, na=False)]
    if not matched.empty:
        print(f'  [{corp_name}] 부분 일치 후보: {matched["corp_name"].tolist()}')
        return matched.iloc[0]['corp_code']
    return None

company_map = {}  # corp_name -> corp_code
for name, stock in TARGET_COMPANIES.items():
    code = get_corp_code(name, stock, corp_df)
    company_map[name] = code
    status = code if code else '못 찾음'
    print(f'{name:20s} corp_code: {status}')

## 3. 공시 목록 수집 함수

In [ ]:
REPORT_TYPES = {
    'A001': '사업보고서',
    'A002': '반기보고서',
    'A003': '분기보고서',
    'F001': '감사보고서',
    'F002': '연결감사보고서',
}

# 6년치 날짜 범위
END_DATE   = datetime.today().strftime('%Y%m%d')
START_DATE = str(int(END_DATE[:4]) - 6) + END_DATE[4:]
print(f'수집 기간: {START_DATE} ~ {END_DATE}')


def fetch_report_list(api_key: str, corp_code: str, report_type: str,
                      bgn_de: str, end_de: str) -> list[dict]:
    """특정 회사의 특정 보고서 공시 목록 반환 (페이지 자동 순회)"""
    base_url = 'https://opendart.fss.or.kr/api/list.json'
    results = []
    page = 1

    while True:
        params = {
            'crtfc_key':        api_key,
            'corp_code':        corp_code,
            'pblntf_detail_ty': report_type,
            'bgn_de':           bgn_de,
            'end_de':           end_de,
            'page_no':          page,
            'page_count':       100,
        }
        resp = requests.get(base_url, params=params, timeout=20)
        resp.raise_for_status()
        data = resp.json()

        if data.get('status') != '000':
            # 000=정상, 013=조회된 데이터가 없음
            break

        results.extend(data.get('list', []))

        total_count = int(data.get('total_count', 0))
        if len(results) >= total_count:
            break
        page += 1

    return results

## 4. 전체 수집 실행

In [ ]:
all_records = []

for corp_name, corp_code in company_map.items():
    if not corp_code:
        print(f'[SKIP] {corp_name}: corp_code 없음')
        continue

    for rtype, rname in REPORT_TYPES.items():
        items = fetch_report_list(API_KEY, corp_code, rtype, START_DATE, END_DATE)
        for item in items:
            item['corp_name_kr'] = corp_name
            item['report_type_name'] = rname
        all_records.extend(items)
        print(f'  {corp_name} | {rname}: {len(items)}건')

print(f'\n총 수집: {len(all_records)}건')

In [ ]:
# API 응답 구조 확인 - 삼양사 사업보고서 1건만 테스트
test_corp_name, test_corp_code = '삼양사', company_map['삼양사']
params = {
    'crtfc_key':        API_KEY,
    'corp_code':        test_corp_code,
    'pblntf_detail_ty': 'A001',
    'bgn_de':           START_DATE,
    'end_de':           END_DATE,
    'page_no':          1,
    'page_count':       1,
}
raw = requests.get('https://opendart.fss.or.kr/api/list.json', params=params, timeout=20).json()

print('=== 응답 최상위 키 ===')
for key, val in raw.items():
    if key != 'list':
        print(f'  {key:15s}: {val}')

print()
print('=== list[0] 필드 (레코드 1건) ===')
if raw.get('list'):
    for key, val in raw['list'][0].items():
        print(f'  {key:20s}: {val}')

## 4-1. 보고서 본문 XML 수집

`rcept_no`로 `/document.xml` 호출 → ZIP 안의 XML 파일을 파싱합니다.  
ZIP 안에는 보고서 섹션별 XML 파일이 여러 개 들어있으며, 확장자 없는 파일이 본문 원본입니다.

In [ ]:
# 단건 테스트 - 삼양사 사업보고서 첫 번째 rcept_no
sample = next(r for r in all_records if r['corp_name_kr'] == '삼양사' and r['report_type_name'] == '사업보고서')
rcept_no = sample['rcept_no']
print(f'테스트 접수번호: {rcept_no}  ({sample["report_nm"]})')

url = f'https://opendart.fss.or.kr/api/document.xml?crtfc_key={API_KEY}&rcept_no={rcept_no}'
resp = requests.get(url, timeout=30)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    file_list = z.namelist()
    print(f'\nZIP 내 파일 목록 ({len(file_list)}개):')
    for fname in file_list:
        print(f'  {fname}')

In [ ]:
from lxml import etree

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    main_file = next(
        (f for f in file_list if '.' not in f),
        max((f for f in file_list if f.endswith('.xml')), key=lambda f: z.getinfo(f).file_size, default=None)
    )
    print(f'본문 파일: {main_file}')
    raw_xml = z.read(main_file)

clean_xml = decode_xml_bytes(raw_xml).encode('utf-8')

parser = etree.XMLParser(recover=True, encoding='utf-8')
root = etree.fromstring(clean_xml, parser=parser)

print(f'\n루트 태그   : {root.tag}')
print(f'직계 자식 수 : {len(list(root))}')
print('\n--- 상위 구조 (depth 2까지) ---')
for child in root:
    print(f'  <{child.tag}>')
    for grandchild in list(child)[:3]:
        text_preview = (grandchild.text or '').strip()[:80]
        print(f'    <{grandchild.tag}> {text_preview}')

In [ ]:
import time
import re

xml_output_dir = '../data/dart_xml'
os.makedirs(xml_output_dir, exist_ok=True)

def decode_xml_bytes(xml_bytes: bytes) -> str:
    """XML 선언의 encoding 속성을 먼저 읽고, 없으면 순서대로 시도"""
    # XML 선언에서 encoding 추출 (앞 200바이트만 확인)
    header = xml_bytes[:200]
    match = re.search(rb'encoding=["\']([^"\']+)["\']', header, re.IGNORECASE)
    candidates = []
    if match:
        candidates.append(match.group(1).decode('ascii'))
    candidates += ['utf-8-sig', 'euc-kr', 'cp949', 'latin-1']

    for enc in candidates:
        try:
            return xml_bytes.decode(enc)
        except (UnicodeDecodeError, LookupError):
            continue
    raise ValueError('디코딩 실패: 알 수 없는 인코딩')

def fetch_and_save_xml(api_key: str, rcept_no: str, save_dir: str) -> str | None:
    """보고서 본문 XML을 받아 utf-8로 변환 후 저장. 이미 있으면 스킵."""
    save_path = os.path.join(save_dir, f'{rcept_no}.xml')
    if os.path.exists(save_path):
        return save_path

    url = f'https://opendart.fss.or.kr/api/document.xml?crtfc_key={api_key}&rcept_no={rcept_no}'
    resp = requests.get(url, timeout=30)
    if resp.status_code != 200:
        return None

    try:
        with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
            names = z.namelist()
            main_file = next(
                (f for f in names if '.' not in f),
                max((f for f in names if f.endswith('.xml')), key=lambda f: z.getinfo(f).file_size, default=None)
            )
            if not main_file:
                return None
            xml_bytes = z.read(main_file)
    except Exception:
        return None

    try:
        clean_bytes = decode_xml_bytes(xml_bytes).encode('utf-8')
    except ValueError:
        return None

    with open(save_path, 'wb') as f:
        f.write(clean_bytes)
    return save_path


success, skip, fail = 0, 0, 0
for i, record in enumerate(all_records):
    rcept_no = record.get('rcept_no')
    if not rcept_no:
        fail += 1
        continue

    if os.path.exists(os.path.join(xml_output_dir, f'{rcept_no}.xml')):
        skip += 1
        continue

    result = fetch_and_save_xml(API_KEY, rcept_no, xml_output_dir)
    if result:
        success += 1
        print(f'  [{i+1:03d}] {record["corp_name_kr"]} | {record["report_type_name"]} | {record["report_nm"]}')
    else:
        fail += 1
        print(f'  [FAIL] {rcept_no}')

    time.sleep(0.3)

print(f'\n완료 - 성공:{success} 스킵:{skip} 실패:{fail}')
print(f'저장 경로: {xml_output_dir}')

## 5. 결과 정리 및 저장

In [ ]:
df = pd.DataFrame(all_records)

keep_cols = [
    'corp_name_kr', 'corp_name', 'corp_code', 'stock_code',
    'report_type_name', 'report_nm',
    'rcept_no', 'rcept_dt', 'flr_nm', 'rm'
]
df = df[[c for c in keep_cols if c in df.columns]]
df['rcept_dt'] = pd.to_datetime(df['rcept_dt'], format='%Y%m%d')
df = df.sort_values(['corp_name_kr', 'rcept_dt']).reset_index(drop=True)

output_dir = '../data'
os.makedirs(output_dir, exist_ok=True)
output_path = f'{output_dir}/dart_reports.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {output_path}  ({len(df)}건)')
df.head(5)

## 6. 누락 보고서 확인

- **상장사** (삼양사·삼양패키징·삼양엔씨켐): 6년치 기준 사업보고서 ~6건, 반기 ~6건, 분기 ~12건 예상  
- **비상장사**: 정기공시 의무 없음 → 감사보고서만 제출하므로 0건이어도 정상  
- 전체 0건인 회사는 corp_code 매핑 오류 가능성 확인 필요

In [ ]:
# 회사별 / 보고서 유형별 건수 피벗
summary = (
    df.groupby(['corp_name_kr', 'report_type_name'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['사업보고서', '반기보고서', '분기보고서', '감사보고서', '연결감사보고서'], fill_value=0)
)
print('=== 수집 건수 요약 ===')
print(summary.to_string())

# 상장사: 사업보고서 기준 6건 미만이면 누락 의심
LISTED = ['삼양사', '삼양패키징', '삼양엔씨켐']
print('\n=== 상장사 누락 진단 (사업보고서 6건 미만) ===')
for corp in LISTED:
    if corp not in summary.index:
        print(f'  [없음] {corp}: 수집 데이터 자체 없음')
        continue
    cnt = summary.loc[corp, '사업보고서']
    flag = '⚠ 누락 의심' if cnt < 6 else '✓ 정상'
    print(f'  {corp:12s} 사업보고서 {cnt}건  {flag}')

# 전체 0건인 회사 → corp_code 매핑 재확인 필요
print('\n=== 전체 0건 회사 (corp_code 확인 필요) ===')
zero_corps = [corp for corp in summary.index if summary.loc[corp].sum() == 0]
all_corps = list(company_map.keys())
missing_in_df = [c for c in all_corps if c not in summary.index]
for corp in zero_corps + missing_in_df:
    ccode = company_map.get(corp, '없음')
    print(f'  {corp:20s} corp_code: {ccode}')
    # DART에서 해당 corp_code로 어떤 회사인지 재확인
    if ccode and ccode in corp_df['corp_code'].values:
        matched = corp_df[corp_df['corp_code'] == ccode].iloc[0]
        print(f'    → DART 등록명: {matched["corp_name"]}  종목코드: {matched["stock_code"]}')

In [ ]:
# 0건 회사의 이름으로 DART에서 직접 검색해 후보 확인
zero_all = zero_corps + missing_in_df
if zero_all:
    print('=== DART corp_df에서 이름 재검색 ===')
    for corp in zero_all:
        candidates = corp_df[corp_df['corp_name'].str.contains(corp.replace('삼양', ''), na=False)]
        candidates = candidates[candidates['corp_name'].str.contains('삼양', na=False)]
        print(f'\n  [{corp}] 후보 목록:')
        print(candidates[['corp_code', 'corp_name', 'stock_code']].to_string(index=False))